# AllSortsHub Narrated Studio (Colab, free-tier)

Keeps the **original flat-cartoon art style** from `studio.py` — Ravi, the same
backgrounds, the same character drawing code — and adds:

- Free neural narration (`edge-tts`) per scene
- Scene duration driven automatically by how long the narration audio is (write more, get more runtime — no manual duration guessing)
- Background music mixing
- The same ffmpeg concat + vertical Shorts pipeline `studio.py` already uses

**No images needed at all.** This notebook imports `studio.py`'s own `frame()`, `character()`,
`bg()` and `wrap()` functions directly from your repo, so every frame is drawn with the exact
same code that made your original video — just narrated and stretched to 5-8 minutes.

## 1. Setup

In [ ]:
!pip -q install pillow
!apt-get -y -qq install ffmpeg > /dev/null

# Free TTS via edge-tts CLI (subprocess-based, avoids asyncio issues in Colab)
!pip -q install edge-tts

import json, os, shutil, subprocess, math, time, sys, importlib.util
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

REPO_URL = "https://github.com/parth01/AllSortsHub-Cartoon-Studio.git"
REPO = Path("/content/AllSortsHub-Cartoon-Studio")
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)

BUILD = Path("/content/build")
OUT = REPO / "output"
MUSIC = REPO / "claude files" / "music"
EP = REPO / "claude files" / "episode_narrated.json"
for p in (BUILD, OUT, MUSIC):
    p.mkdir(parents=True, exist_ok=True)

print("Repo ready at", REPO)

## 2. Import the ORIGINAL drawing code from studio.py

This loads `studio.py` as a module so we reuse `frame()`, `character()`, `bg()`, `wrap()`, `font()`
directly — the exact same visuals as your existing episode. If you ever tweak `studio.py`'s art
(new mood, new background), this notebook picks it up automatically next run.

In [ ]:
spec = importlib.util.spec_from_file_location("studio", REPO / "studio.py")
studio = importlib.util.module_from_spec(spec)
spec.loader.exec_module(studio)

W, H, FPS = studio.W, studio.H, studio.FPS
print(f"Loaded studio.py — canvas {W}x{H} @ {FPS}fps")

## 3. Write your episode script

Same scene format as `episode_01.json` (background / characters / title / prop), plus one new
field: `"narration"`. Duration is now computed automatically from how long that narration takes
to speak — delete any `"duration"` field, it's ignored. `"subtitle"` defaults to the narration
text itself if you don't set one separately.

Edit `episode_narrated.json` (create it in `claude files/` in your repo, or run the cell below to
write a 3-scene starter you can expand).

In [ ]:
starter = {
  "title": "AllSortsHub_Episode_01_Narrated",
  "scenes": [
    {
      "background": "bedroom",
      "characters": [{"name": "Ravi", "x": 960, "y": 450, "mood": "normal"}],
      "title": "The Guy Who Got $1 Billion for One Day",
      "narration": "Ravi woke up on an ordinary Tuesday, completely unaware that his whole life was about to change before lunch."
    },
    {
      "background": "bedroom",
      "characters": [{"name": "Ravi", "x": 960, "y": 450, "mood": "shocked"}],
      "prop": "$1,000,000,000",
      "narration": "On the kitchen table sat a plain envelope with no return address, and inside it was a single typed sentence that made no sense at all."
    },
    {
      "background": "bank",
      "characters": [{"name": "Ravi", "x": 960, "y": 500, "mood": "shocked"}],
      "narration": "By the time he reached the bank, a line of reporters was already waiting outside, and none of them would say why."
    }
  ]
}

if not EP.exists():
    EP.write_text(json.dumps(starter, indent=2))
    print("Wrote starter script to", EP)
else:
    print("Using existing script at", EP)

print(EP.read_text())

## 4. Free narration via edge-tts (subprocess, with retries)

In [ ]:
def tts(text, voice, out_path, retries=3):
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            result = subprocess.run(
                ["edge-tts", "--voice", voice, "--text", text, "--write-media", str(out_path)],
                capture_output=True, text=True, timeout=60,
            )
            if Path(out_path).exists() and Path(out_path).stat().st_size > 0:
                return
            last_err = result.stderr or "empty output file"
        except Exception as e:
            last_err = e
        time.sleep(1.5 * attempt)
    raise RuntimeError(f"edge-tts failed after {retries} attempts for voice={voice!r}, text={text[:50]!r}: {last_err}")

def audio_duration(path):
    result = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "default=noprint_wrappers=1:nokey=1", str(path)],
        capture_output=True, text=True,
    )
    out = result.stdout.strip()
    if not out:
        raise RuntimeError(f"ffprobe couldn't read duration for {path} (stderr: {result.stderr.strip()})")
    return float(out)

# Good free voices: en-US-GuyNeural, en-US-AriaNeural, en-GB-RyanNeural, en-US-JennyNeural

## 5. Build the episode using studio.py's own frame() function

In [ ]:
def build_episode(ep_path, voice_default="en-US-GuyNeural"):
    data = json.loads(Path(ep_path).read_text())
    scenes = data["scenes"]
    scene_audio, scene_video = [], []

    for i, s in enumerate(scenes, 1):
        print(f"Scene {i}/{len(scenes)}: generating narration...")
        text = s["narration"]
        voice = s.get("voice", voice_default)
        a_path = BUILD / f"a{i:02d}.mp3"
        tts(text, voice, a_path)
        dur = audio_duration(a_path) + 0.5  # small padding

        scene_for_frame = dict(s)
        scene_for_frame.setdefault("subtitle", text)
        scene_audio.append(a_path)

        scene_dir = BUILD / f"s{i:02d}"
        scene_dir.mkdir(exist_ok=True)
        n_frames = max(1, int(dur * FPS))
        for j in range(n_frames):
            img = studio.frame(scene_for_frame, j / FPS)
            img.save(scene_dir / f"f{j:05d}.jpg", quality=90)

        v_path = BUILD / f"v{i:02d}.mp4"
        subprocess.run(
            ["ffmpeg", "-y", "-framerate", str(FPS), "-i", str(scene_dir / "f%05d.jpg"),
             "-c:v", "libx264", "-pix_fmt", "yuv420p", "-preset", "veryfast", "-crf", "19", str(v_path)],
            check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        scene_video.append(v_path)
        print(f"  rendered ({dur:.1f}s)")

    vlist = BUILD / "v_list.txt"
    vlist.write_text("".join(f"file '{p.as_posix()}'\n" for p in scene_video))
    silent = BUILD / "silent.mp4"
    subprocess.run(["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", str(vlist), "-c", "copy", str(silent)], check=True)

    alist = BUILD / "a_list.txt"
    alist.write_text("".join(f"file '{p.as_posix()}'\n" for p in scene_audio))
    narration = BUILD / "narration.mp3"
    subprocess.run(["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", str(alist), "-c", "copy", str(narration)], check=True)

    final_audio = narration
    music_files = list(MUSIC.glob("*.mp3"))
    if music_files:
        mixed = BUILD / "mixed.mp3"
        subprocess.run([
            "ffmpeg", "-y", "-i", str(narration), "-stream_loop", "-1", "-i", str(music_files[0]),
            "-filter_complex", "[1:a]volume=0.15[m];[0:a][m]amix=inputs=2:duration=first:dropout_transition=2[aout]",
            "-map", "[aout]", str(mixed),
        ], check=True)
        final_audio = mixed

    master = OUT / f"{data.get('title', 'episode')}.mp4"
    subprocess.run([
        "ffmpeg", "-y", "-i", str(silent), "-i", str(final_audio),
        "-c:v", "copy", "-c:a", "aac", "-shortest", "-movflags", "+faststart", str(master),
    ], check=True)
    total_scenes_time = sum(audio_duration(a) for a in scene_audio)
    print(f"\nMaster episode created: {master}")
    print(f"Total narration length: {total_scenes_time/60:.1f} minutes")
    return master

## 6. Vertical Shorts from the finished master

In [ ]:
def make_shorts(master_path, highlights):
    for k, (start, dur) in enumerate(highlights, 1):
        out = OUT / f"{master_path.stem}_Short_{k:02d}.mp4"
        subprocess.run([
            "ffmpeg", "-y", "-ss", str(start), "-t", str(dur), "-i", str(master_path),
            "-vf", "scale=-2:1920,crop=1080:1920:(in_w-1080)/2:0,format=yuv420p",
            "-c:v", "libx264", "-preset", "veryfast", "-crf", "21", "-c:a", "aac", "-movflags", "+faststart", str(out),
        ], check=True)
    print("Shorts created in", OUT)

## 7. Run it

In [ ]:
master = build_episode(EP)
# Pick real timestamps after watching the master once:
make_shorts(master, [(5, 20), (30, 20)])

## 8. Reaching 5-8 minutes

Runtime = total time it takes to speak every scene's `"narration"` text aloud (~150 words/minute).
For 5-8 minutes you need roughly **800-1200 words** of narration split across 20-30 scenes.

Just keep adding scene objects to `episode_narrated.json` — reuse `"background"` values already
defined in `studio.py` (`bedroom`, `city`, `bank`, or anything else falls back to the default street
scene) and `"mood"` values (`normal`, `happy`, `shocked`). No new art assets required; it's the
same code drawing every scene.

**Push this notebook to your repo** the same way as before: GitHub → your repo → Add file →
Upload files → into `claude files/`.